In [4]:
%pip install pandas
import pandas as pd

ModuleNotFoundError: No module named 'pandas'

In [ ]:
stats = pd.read_csv('data/advanced_stats.csv')
salary = pd.read_csv('data/player_salary.csv')

In [ ]:
# -------- Clean advanced stats --------
stats_clean = stats.copy()

# Drop ranking/helper columns we do not need for modeling
stats_clean = stats_clean.drop(columns=["Rk", "Player-additional"], errors="ignore")

# Remove duplicate player rows and fully empty columns
stats_clean = stats_clean.drop_duplicates(subset=["Player", "Team"], keep="first")
stats_clean = stats_clean.dropna(axis=1, how="all")

# Fill awards with empty string so it is consistent text data
if "Awards" in stats_clean.columns:
    stats_clean["Awards"] = stats_clean["Awards"].fillna("")

# Standardize text columns
for col in ["Player", "Team", "Pos"]:
    if col in stats_clean.columns:
        stats_clean[col] = stats_clean[col].astype(str).str.strip()


# -------- Clean salary table --------
# The first row contains true column names and the file has unnamed placeholders
salary_clean = salary.copy()

# Rename known placeholder columns first
salary_clean = salary_clean.rename(
    columns={
        "Unnamed: 0": "Rk",
        "Unnamed: 1": "Player",
        "Unnamed: 2": "Team",
    }
)

# Drop the repeated header row inside the data
salary_clean = salary_clean[salary_clean["Rk"].astype(str) != "Rk"].copy()

# Rename salary columns to readable season names if present
season_map = {
    "Salary": "2025-26",
    "Salary.1": "2026-27",
    "Salary.2": "2027-28",
    "Salary.3": "2028-29",
    "Salary.4": "2029-30",
    "Salary.5": "2030-31",
    "Unnamed: 9": "Guaranteed",
    "-additional": "player_id",
}
salary_clean = salary_clean.rename(columns={k: v for k, v in season_map.items() if k in salary_clean.columns})

# Keep only useful columns if they exist
keep_cols = [
    "Rk", "Player", "Team",
    "2025-26", "2026-27", "2027-28", "2028-29", "2029-30", "2030-31",
    "Guaranteed", "player_id",
]
salary_clean = salary_clean[[c for c in keep_cols if c in salary_clean.columns]]

# Parse currency strings to numeric (e.g. "$59,606,817" -> 59606817)
money_cols = [c for c in salary_clean.columns if c.startswith("20") or c == "Guaranteed"]
for col in money_cols:
    salary_clean[col] = (
        salary_clean[col]
        .astype("string")
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .replace({"": pd.NA})
    )
    salary_clean[col] = pd.to_numeric(salary_clean[col], errors="coerce")

# Rank should be numeric
if "Rk" in salary_clean.columns:
    salary_clean["Rk"] = pd.to_numeric(salary_clean["Rk"], errors="coerce")

# Normalize player names first, then drop missing/blank players
if "Player" in salary_clean.columns:
    salary_clean["Player"] = salary_clean["Player"].astype("string").str.strip()
    salary_clean = salary_clean[salary_clean["Player"].notna() & (salary_clean["Player"] != "")]

# Basic text cleanup without converting nulls to literal "nan"
for col in ["Team", "player_id"]:
    if col in salary_clean.columns:
        salary_clean[col] = salary_clean[col].astype("string").str.strip()


# -------- Save cleaned datasets --------
stats_clean.to_csv("data/advanced_stats_clean.csv", index=False)
salary_clean.to_csv("data/player_salary_clean.csv", index=False)

print("Advanced stats cleaned shape:", stats_clean.shape)
print("Salary cleaned shape:", salary_clean.shape)
print("\nAdvanced stats preview:")
print(stats_clean.head())
print("\nSalary preview:")
print(salary_clean.head())